### KNN From Scratch

In [62]:
from sklearn.datasets import load_iris, load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
import numpy as np
from collections import Counter

In [ ]:
class KNN:
    def __init__(self, k, metric='euclidean'):
        self.k=k
        
    def euclidean(self,x1,x2):
        total = 0
        for feature1, feature2 in zip(x1,x2):
            total+=(feature1-feature2)**2
        return np.sqrt(total)
    
    def get_k_neighbors(self,test_row, n_train_samples):
        full_arr = []
        for i in range(n_train_samples):
            dist = self.euclidean(self.X_train[i],test_row)
            full_arr.append((self.X_train[i], self.y_train[i],dist))
        full_arr.sort(key= lambda tup: tup[2])
        return [(tup[0],tup[1]) for tup in full_arr[:self.k]]
    
    def _predict_classify(self, test_row):
        neighbors = self.get_k_neighbors(test_row=test_row,n_train_samples=self.X_train.shape[0])
        y_train_neighbors = [tup[1] for tup in neighbors]
        mode = Counter(y_train_neighbors).most_common(1)[0][0]
        return mode
    
    def _predict_regression(self, test_row):
        neighbors = self.get_k_neighbors(test_row=test_row,n_train_samples=self.X_train.shape[0])
        y_train_neighbors = [tup[1] for tup in neighbors]
        return np.mean(y_train_neighbors)
    
    def predict_classify(self, X_test):
        y_pred = []
        for test in X_test:
            y_pred.append(self._predict_classify(test))
        return y_pred
    
    def predict_regression(self, X_test):
        y_pred = []
        for test in X_test:
            y_pred.append(self._predict_regression(test))
        return y_pred
    
    def fit(self,X,y):
        self.X_train = X
        self.y_train = y

### KNN for classification

In [69]:
knn = KNN(k=2,metric='euclidean')
data = load_iris()
X,y=data.data, data.target

feature_names = data.feature_names
target_names = data.target_names
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.4,random_state=42)

scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
knn.fit(X_train,y_train)
y_pred = knn.predict_classify(X_test)
print(classification_report(y_test,y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        23
           1       0.95      0.95      0.95        19
           2       0.94      0.94      0.94        18

    accuracy                           0.97        60
   macro avg       0.96      0.96      0.96        60
weighted avg       0.97      0.97      0.97        60



### KNN using sklearn

In [70]:
clf = KNeighborsClassifier(metric='euclidean',n_neighbors=3)
clf.fit(X_train,y_train)
y_pred_sklearn = clf.predict(X_test)
print(classification_report(y_test,y_pred_sklearn))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        23
           1       0.95      1.00      0.97        19
           2       1.00      0.94      0.97        18

    accuracy                           0.98        60
   macro avg       0.98      0.98      0.98        60
weighted avg       0.98      0.98      0.98        60



### KNN for regression

In [76]:
data = load_diabetes()
X,y=data.data, data.target

feature_names = data.feature_names

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.4,random_state=42)

knn.fit(X_train,y_train)
y_pred = knn.predict_regression(X_test)
print(r2_score(y_test,y_pred))

0.32919832554066797


### Learn the scaling parameters only from the training data - WHY??

- Never let your test data influence how your model or preprocessing is built — it should only be used for final evaluation.
- This causes data leakage, meaning your model gets access to information it shouldn’t have, and your evaluation metrics (accuracy, precision, etc.) become artificially inflated and unreliable.

### KNN using sklearn

In [77]:
clf = KNeighborsRegressor(metric='euclidean')
clf.fit(X_train,y_train)
y_pred_sklearn = clf.predict(X_test)
print(r2_score(y_test,y_pred_sklearn))

0.3792146528837278
